# Massive multistart — screen a huge candidate set, Adam-refine the survivors

For every `h`, cover the angle space with an enormous set of starting points, run Adam from them,
and keep the best endpoint. No model, no labels — just search.

**The honest arithmetic first.** Running Adam from 10^11 starts per instance is not a large run,
it is an impossible one: 500 instances x 10^11 starts x 500 Adam steps is 2.5x10^16 gradient
evaluations, and a T4 does ~10^5 of them per second. That is roughly 8 million years. Even
*evaluating* each start once, with no optimisation at all, would take about a decade.

What actually fits in a Kaggle session is **10^5 - 10^6 starts per instance**, and the way to buy
the most quality with them is not to spend them uniformly:

| stage | what it does | cost per point | points per instance |
|---|---|---|---|
| **screen** | evaluate `P(ground)` once, no gradient | 1x | ~65 000 |
| **coarse** | 60 Adam steps on the screen survivors | ~180x | 256 |
| **fine** | 400 Adam steps on the coarse survivors | ~1200x | 32 |

A forward-only evaluation is ~3x cheaper than a gradient step, so screening 65 000 candidates
costs about the same as Adam-refining 40 of them. Spending that budget on breadth first, then
narrowing, covers vastly more of the landscape than 32 random restarts ever could — and §4 checks
that claim against exactly that baseline instead of assuming it.

**Why Sobol and not a grid.** "Evenly spaced" in 10 dimensions cannot be done with a grid: even
4 points per axis is 4^10 = 1.05M points, and 10^11 points would still be only 12 per axis —
a 0.5 rad resolution, coarser than the basins we are hunting. A scrambled **Sobol sequence** is
the right generalisation: it is low-discrepancy (evenly spaced in the sense that matters, with no
axis alignment), works at any count, and every prefix of it is itself well spread, so `n_sobol`
is a smooth quality knob rather than a 10th-root cliff.

**Box.** `beta` is exactly pi-periodic, so `(-pi/2, pi/2)` is one full period and nothing is lost.
`gamma` has no exact period; a probe over `(-pi, pi)` found the top-1% of samples using the full
width (90th pct of `|gamma|` = 2.63, 99th = 3.06), so the box is not oversized.

## 0. Setup

In [ ]:
import os, sys, glob, time, math, csv, json
import numpy as np
import torch
from torch.quasirandom import SobolEngine
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
else:
    print("  WARNING: no GPU. Set QUICK = True below or this will take hours.")

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)


def find_file(name):
    """Locate an organiser file across Kaggle input, Colab, or a local checkout."""
    for root in ["/kaggle/input", "/kaggle/working", "data/raw", "../data/raw", ".", "..", "/content"]:
        if os.path.isdir(root):
            hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
            if hits:
                return hits[0]
    raise FileNotFoundError(
        f"Could not find {name}. Attach the competition files as a Kaggle dataset "
        f"(J.npy, h_train.npy, QAOA.py) or put them in ./data/raw/.")


QAOA_PATH = find_file("QAOA.py")
sys.path.insert(0, os.path.dirname(os.path.abspath(QAOA_PATH)))
from QAOA import QAOA, P as P_DEPTH, N_QUBITS

J = np.load(find_file("J.npy")).astype(np.float64)
h_train = np.load(find_file("h_train.npy")).astype(np.float64)
qaoa = QAOA(torch.tensor(J, dtype=torch.float32), device=DEVICE)
print(f"\nJ {J.shape} | h_train {h_train.shape} | p={P_DEPTH} | n={N_QUBITS}")

## 1. Configuration

Every knob is a point on the same trade-off: breadth (`n_sobol`) against depth (`steps_fine`).
The cell prints the resulting work in gradient-equivalent units so you can size a run before
starting it — a forward-only screen counts as 1/3 of a gradient step.

- **`n_sobol`** — candidates screened per instance. The dominant cost and the whole point of the
  notebook. Scale this until the session budget is used up.
- **`keep_screen` / `keep_coarse`** — funnel widths. Too narrow and the screen's breadth is thrown
  away before Adam can exploit it; §4 shows the shape of that trade-off.
- **`*_rows`** — GPU memory only. Screening runs under `no_grad` so it holds ~3 live
  `(rows, 4096)` complex tensors (16384 rows ≈ 1.6 GiB); Adam stores ~65 of them
  (2048 rows ≈ 4.4 GiB). Halve either on OOM.

In [ ]:
CFG = dict(
    n_sobol      = 65536,   # Sobol candidates screened per instance
    n_ramp       = 2048,    # adiabatic linear-ramp family added to the pool
    keep_screen  = 256,     # survivors of the screen, per instance
    steps_coarse = 60,      # Adam steps on those
    keep_coarse  = 32,      # survivors of the coarse pass
    steps_fine   = 400,     # Adam steps on those
    lr           = 0.08,    # coarse lr (cosine-annealed to lr/25)
    lr_fine      = 0.05,
    elite_from   = 64,      # instances solved first to harvest transfer starts (0 = off)
    elite_per    = 4,       # best angles taken from each of them
    screen_rows  = 16384,   # (instance x candidate) rows on the GPU at once, no_grad
    adam_rows    = 2048,    # ditto, with autograd
)
QUICK = False               # True -> tiny run to check the notebook end to end
if QUICK:
    CFG.update(n_sobol=2048, n_ramp=256, keep_screen=32, steps_coarse=20,
               keep_coarse=8, steps_fine=60, elite_from=8)

GAMMA_MAX = math.pi         # gamma box; beta is fixed to one period, (-pi/2, pi/2)

n_pool = CFG["n_sobol"] + CFG["n_ramp"]
work = dict(
    screen = 500 * n_pool / 3,
    coarse = 500 * CFG["keep_screen"] * CFG["steps_coarse"],
    fine   = 500 * CFG["keep_coarse"] * CFG["steps_fine"],
)
print(json.dumps(CFG, indent=2))
print(f"\ncandidate pool: {n_pool} points per instance")
print("gradient-equivalent work for 500 instances:")
for k, v in work.items():
    print(f"  {k:7s}: {v/1e6:8.2f}M  ({v/sum(work.values())*100:4.1f}%)")
print(f"  {'total':7s}: {sum(work.values())/1e6:8.2f}M")

## 2. The candidate pool

Three families, concatenated:

1. **Sobol** — low-discrepancy coverage of the box, the bulk of the pool.
2. **Linear ramp** — a discretised adiabatic schedule (`gamma` rising, `beta` falling). Notebook 02
   found this family lands in the good basin far more often than chance; it is cheap insurance
   that the pool contains at least a few strong points even if Sobol misses.
3. **Elite transfer** *(optional)* — angles that already won on *other* instances. Good angles for
   this fixed `J` cluster across instances, so the winners from a first pass over `elite_from`
   instances are unusually strong starts for the remaining ones. This is the one part of the
   search that is not memoryless, and it costs nothing extra to evaluate.

The harvest instances are drawn from `h_train` and stay in the main run. That is not leakage:
this is a search, not an estimator — every angle it reports was found and verified by the
simulator for that specific instance, so reusing a solution it already has is legitimate. For
`h_test` the same pool is a genuine transfer across instances.

In [ ]:
def sobol_pool(n, seed=0):
    """Scrambled Sobol points in the angle box. Low-discrepancy => 'evenly spaced' in 10-D."""
    u = SobolEngine(2 * P_DEPTH, scramble=True, seed=seed).draw(n).numpy().astype(np.float64)
    g = (u[:, :P_DEPTH] * 2 - 1) * GAMMA_MAX
    b = (u[:, P_DEPTH:] * 2 - 1) * (np.pi / 2)     # exactly one period of beta
    return np.concatenate([g, b], axis=1)


def ramp_pool(n, gen):
    """Discretised adiabatic schedules with jitter: gamma ramps up, beta ramps down."""
    dt = gen.uniform(0.2, 1.4, (n, 1))
    l = np.arange(P_DEPTH)[None, :]
    g = (l + 1) / P_DEPTH * dt
    b = (1 - l / P_DEPTH) * dt
    a = np.concatenate([g, b], axis=1)
    return a + gen.normal(0, 0.15, a.shape)


def build_pool(cfg, seed=0, elite=None):
    gen = np.random.default_rng(seed)
    parts = [sobol_pool(cfg["n_sobol"], seed)]
    if cfg["n_ramp"]:
        parts.append(ramp_pool(cfg["n_ramp"], gen))
    if elite is not None and len(elite):
        parts.append(elite)
    pool = np.concatenate(parts, axis=0)
    return torch.tensor(pool, dtype=torch.float32, device=DEVICE)

## 3. Screen, then refine

`screen` is the cheap breadth pass: no autograd, every candidate scored once, top-`keep` kept per
instance. `refine` is batched Adam over a per-instance set of candidates — the same optimiser as
notebook 02, just handed better starting points and run as a funnel.

Both flatten `(instance x candidate)` into one batch dimension, so the GPU optimises every
instance and every candidate simultaneously; the chunk sizes only bound memory.

In [ ]:
@torch.no_grad()
def screen(h, pool, keep, chunk_rows, log=True):
    """Score every pool candidate against every instance; keep the best `keep` per instance.

    h: (N, 12) tensor on DEVICE. pool: (M, 10). Returns (cand (N, keep, 10), p (N, keep)).
    """
    n, m_pool = len(h), len(pool)
    m = min(m_pool, chunk_rows)
    n_h = max(1, chunk_rows // m)
    cand = torch.empty(n, keep, 2 * P_DEPTH, device=DEVICE)
    best = torch.empty(n, keep, device=DEVICE)
    t0 = time.time()

    for s in range(0, n, n_h):
        hc = h[s:s + n_h]
        c = len(hc)
        scores = torch.empty(c, m_pool, device=DEVICE)
        for lo in range(0, m_pool, m):
            sl = pool[lo:lo + m]
            mm = len(sl)
            hb = hc.repeat_interleave(mm, 0)
            ab = sl.repeat(c, 1)
            p = qaoa.p_ground(hb, ab[:, :P_DEPTH], ab[:, P_DEPTH:])
            scores[:, lo:lo + mm] = p.view(c, mm)
        v, idx = scores.topk(keep, dim=1)
        best[s:s + c] = v
        cand[s:s + c] = pool[idx]
        if log and (s // max(n_h, 1)) % 20 == 0:
            done = min(s + n_h, n)
            print(f"  screen {done}/{n}  ({(time.time()-t0)/done*n:.0f}s projected)", flush=True)
    return cand, best


def refine(h, cand, steps, lr, chunk_rows, keep, log=True, tag=""):
    """Adam-ascend every candidate of every instance; keep the best `keep` per instance.

    Each candidate is compared against where it started and the better of the two is kept, so a
    stage can never score worse than its input. Without this the funnel does lose ground: an
    already-good point can be kicked out of its basin by the first few high-lr steps and not find
    its way back inside the stage's budget.

    cand: (N, K, 10). Returns (cand (N, keep, 10), p (N, keep)), sorted best-first.
    """
    n, k, d = cand.shape
    n_h = max(1, chunk_rows // k)
    out_c = torch.empty(n, keep, d, device=DEVICE)
    out_p = torch.empty(n, keep, device=DEVICE)
    t0 = time.time()

    for s in range(0, n, n_h):
        hc = h[s:s + n_h]
        c = len(hc)
        hb = hc.repeat_interleave(k, 0)
        a0 = cand[s:s + c].reshape(c * k, d).clone()
        with torch.no_grad():
            p0 = qaoa.p_ground(hb, a0[:, :P_DEPTH], a0[:, P_DEPTH:])

        a = a0.clone().requires_grad_(True)
        opt = torch.optim.Adam([a], lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr / 25)
        for _ in range(steps):
            opt.zero_grad()
            (-qaoa.p_ground(hb, a[:, :P_DEPTH], a[:, P_DEPTH:]).sum()).backward()
            opt.step()
            sched.step()
        with torch.no_grad():
            p1 = qaoa.p_ground(hb, a[:, :P_DEPTH], a[:, P_DEPTH:])

        keep_new = (p1 >= p0).unsqueeze(1)
        a_best = torch.where(keep_new, a.detach(), a0).view(c, k, d)
        p = torch.maximum(p1, p0).view(c, k)

        v, idx = p.topk(keep, dim=1)
        out_p[s:s + c] = v
        out_c[s:s + c] = a_best.gather(1, idx.unsqueeze(-1).expand(c, keep, d))
        if log and (s // max(n_h, 1)) % 10 == 0:
            done = min(s + n_h, n)
            print(f"  {tag} {done}/{n}  ({(time.time()-t0)/done*n:.0f}s projected)", flush=True)
    return out_c, out_p


def solve(h_np, cfg, seed=0, elite=None, log=True):
    """Full funnel for a batch of instances. Returns (gamma, beta, p, stage_means)."""
    h = torch.tensor(h_np, dtype=torch.float32, device=DEVICE)
    pool = build_pool(cfg, seed, elite)
    stages = {}

    cand, p = screen(h, pool, cfg["keep_screen"], cfg["screen_rows"], log)
    stages["screen"] = p[:, 0].mean().item()

    cand, p = refine(h, cand, cfg["steps_coarse"], cfg["lr"], cfg["adam_rows"],
                     cfg["keep_coarse"], log, "coarse")
    stages["coarse"] = p[:, 0].mean().item()

    cand, p = refine(h, cand, cfg["steps_fine"], cfg["lr_fine"], cfg["adam_rows"],
                     1, log, "fine")
    stages["fine"] = p[:, 0].mean().item()

    best = cand[:, 0].cpu().numpy()
    return best[:, :P_DEPTH], best[:, P_DEPTH:], p[:, 0].cpu().numpy(), stages


def score(h_np, gamma, beta, chunk=2048):
    """Re-score with the organisers' simulator, chunked."""
    out = []
    for s in range(0, len(h_np), chunk):
        hb = torch.tensor(h_np[s:s + chunk], dtype=torch.float32, device=DEVICE)
        g = torch.tensor(gamma[s:s + chunk], dtype=torch.float32, device=DEVICE)
        b = torch.tensor(beta[s:s + chunk], dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            out.append(qaoa.p_ground(hb, g, b).cpu().numpy())
    return np.concatenate(out)

## 4. Does the screen actually buy anything?

The premise is that 65 000 screened starts beat 32 random ones at the *same Adam budget*. That is
testable, and worth testing before committing a long run to it — screening is only useful if the
best point of a huge cheap sample sits in a better basin than a random start does, which is not
guaranteed when Adam can travel a long way from where it began.

Arm A is this notebook's funnel. Arm B is notebook 02's baseline: uniform random starts, same
number of Adam steps, same optimiser. Both end up refining `keep_coarse` candidates, so the
expensive part of the two arms is matched and only the *choice* of starting points differs.

In [ ]:
N_PROBE = 24 if not QUICK else 6
hp = h_train[:N_PROBE]
K = CFG["keep_coarse"]

# --- arm A: screened starts -------------------------------------------------
t0 = time.time()
_, _, pA, stagesA = solve(hp, CFG, seed=1, log=False)
tA = time.time() - t0

# --- arm B: uniform random starts, identical Adam budget --------------------
gen = np.random.default_rng(1)
h_t = torch.tensor(hp, dtype=torch.float32, device=DEVICE)
randc = torch.tensor(
    np.concatenate([gen.uniform(-GAMMA_MAX, GAMMA_MAX, (N_PROBE, K, P_DEPTH)),
                    gen.uniform(-np.pi/2, np.pi/2, (N_PROBE, K, P_DEPTH))], axis=2),
    dtype=torch.float32, device=DEVICE)
t0 = time.time()
_, pB_all = refine(h_t, randc, CFG["steps_coarse"] + CFG["steps_fine"], CFG["lr"],
                   CFG["adam_rows"], K, log=False, tag="rand")
pB = pB_all[:, 0].cpu().numpy()
tB = time.time() - t0

print(f"screened funnel : mean P = {pA.mean():.4f}   ({tA:.0f}s)")
print(f"  stage means   : " + "  ".join(f"{k} {v:.4f}" for k, v in stagesA.items()))
print(f"random restarts : mean P = {pB.mean():.4f}   ({tB:.0f}s)   [notebook 02's method]")
print(f"\nscreening is worth {pA.mean()/max(pB.mean(),1e-12):.2f}x  "
      f"({'keep it' if pA.mean() > pB.mean() else 'NOT worth it — drop n_sobol, raise steps_fine'})")

# returns on pool size: rescreen with Sobol prefixes (every prefix is still well spread)
sizes = [s for s in [256, 1024, 4096, 16384, 65536] if s <= CFG["n_sobol"]]
curve = []
pool_full = build_pool(CFG, 1)
for s in sizes:
    _, pv = screen(h_t, pool_full[:s], 1, CFG["screen_rows"], log=False)
    curve.append(pv[:, 0].mean().item())
    print(f"  best of {s:6d} screened (pre-Adam): {curve[-1]:.4f}")

plt.figure(figsize=(5.5, 3.2))
plt.semilogx(sizes, curve, "o-")
plt.xlabel("Sobol candidates screened"); plt.ylabel("best P(ground), before Adam")
plt.title("returns on breadth"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5. Harvest an elite pool

Solve a small slice of `h_train` first and keep the best angles found. For a fixed `J` the good
basins sit in similar places across instances, so these are strong starts everywhere — they enter
the pool for the main run alongside Sobol and the ramp family.

Set `CFG['elite_from'] = 0` to skip this; the pipeline is unchanged apart from a smaller pool.

In [ ]:
elite = None
if CFG["elite_from"]:
    idx = rng.choice(len(h_train), CFG["elite_from"], replace=False)
    t0 = time.time()
    h_e = torch.tensor(h_train[idx], dtype=torch.float32, device=DEVICE)
    cand, _ = screen(h_e, build_pool(CFG, SEED), CFG["keep_screen"], CFG["screen_rows"], log=False)
    cand, pe = refine(h_e, cand, CFG["steps_coarse"] + CFG["steps_fine"], CFG["lr"],
                      CFG["adam_rows"], CFG["elite_per"], log=False, tag="elite")
    elite = cand.reshape(-1, 2 * P_DEPTH).cpu().numpy().astype(np.float64)
    print(f"harvested {len(elite)} elite starts from {CFG['elite_from']} instances "
          f"in {time.time()-t0:.0f}s  (their own mean P = {pe[:, 0].mean().item():.4f})")
else:
    print("elite transfer disabled")

## 6. Run it on all of `h_train`

This is the number the main-stage leaderboard reports, so it is directly comparable to notebook
02's baseline and to the transformer's validation score.

In [ ]:
t0 = time.time()
gamma, beta, v, stages = solve(h_train, CFG, seed=SEED, elite=elite)
elapsed = time.time() - t0

p = score(h_train, gamma, beta)          # re-scored with the organisers' simulator
assert np.abs(p - v).max() < 1e-4, "optimiser and scorer disagree"

print(f"\n{'='*58}")
print(f"  MASSIVE MULTISTART  (n={len(h_train)} h_train instances)")
print(f"{'='*58}")
for k, val in stages.items():
    print(f"  after {k:6s}     : {val:.5f}")
print(f"  {'-'*54}")
print(f"  mean   P(ground) : {p.mean():.5f}   <- the score")
print(f"  median P(ground) : {np.median(p):.5f}")
print(f"  min / max        : {p.min():.5f} / {p.max():.5f}")
print(f"  random-angle floor: {1/2**N_QUBITS:.5f}")
print(f"  wall clock       : {elapsed:.0f}s  ({elapsed/len(h_train)*1000:.0f} ms/instance)")
print(f"{'='*58}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(p, bins=50)
ax[0].axvline(p.mean(), c="r", ls="--", label=f"mean {p.mean():.4f}")
ax[0].set_xlabel("P(ground)"); ax[0].set_ylabel("count"); ax[0].legend()
ax[0].set_title("per instance")
ax[1].bar(range(len(stages)), list(stages.values()), tick_label=list(stages))
ax[1].set_ylabel("mean P(ground)"); ax[1].set_title("funnel stages")
plt.tight_layout(); plt.show()

## 7. Inference-budget check

The rules give 10 minutes to produce angles for all 500 `h_test` instances. This search *is* the
inference, so the wall clock above is the cost that counts — and a full-breadth run will not fit.
The cell reports the largest `n_sobol` that does, which is the setting to use for a
submission produced by this notebook alone.

In [ ]:
proj = elapsed / len(h_train) * 500
print(f"measured        : {elapsed/len(h_train)*1000:.1f} ms / instance")
print(f"projected (500) : {proj:.0f}s of a 600s budget  ({proj/600*100:.0f}% used)")

if proj > 600:
    head = 600 / proj
    screen_share = work["screen"] / sum(work.values())
    print(f"\nOVER BUDGET by {proj/600:.1f}x. Options, cheapest quality loss first:")
    print(f"  - n_sobol {CFG['n_sobol']} -> {max(256, int(CFG['n_sobol']*head)):d} "
          f"(screening is {screen_share*100:.0f}% of the work; §4's curve shows what this costs)")
    print(f"  - keep_screen {CFG['keep_screen']} -> {max(8, int(CFG['keep_screen']*head)):d}")
    print(f"  - steps_fine {CFG['steps_fine']} -> {max(50, int(CFG['steps_fine']*head)):d}")
    print("\nOr treat this notebook as an offline label generator (§9) and submit a model.")
else:
    print(f"\nWithin budget with {600-proj:.0f}s to spare — room for ~{600/proj:.1f}x more "
          f"breadth (n_sobol -> {int(CFG['n_sobol']*600/proj)}).")

## 8. Submission

Uses `h_test.npy` as soon as it is present, otherwise `h_train.npy` so the cell stays runnable.
Angles are searched for whichever file is loaded — nothing is reused from the run above.

In [ ]:
def write_submission(path, gamma, beta):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id"] + [f"gamma_{i}" for i in range(P_DEPTH)]
                          + [f"beta_{i}" for i in range(P_DEPTH)])
        for i, (gi, bi) in enumerate(zip(gamma, beta)):
            w.writerow([i] + [f"{x:.8f}" for x in gi] + [f"{x:.8f}" for x in bi])


try:
    h_sub, src = np.load(find_file("h_test.npy")).astype(np.float64), "h_test.npy"
    t0 = time.time()
    g_sub, b_sub, _, _ = solve(h_sub, CFG, seed=SEED, elite=elite)
    infer_time = time.time() - t0
except FileNotFoundError:
    h_sub, src = h_train, "h_train.npy (h_test not released yet)"
    g_sub, b_sub, infer_time = gamma, beta, elapsed

write_submission("submission.csv", g_sub, b_sub)
p_sub = score(h_sub, g_sub, b_sub)

print(f"wrote submission.csv from {src}")
print(f"  rows           : {len(h_sub)}")
print(f"  mean P(ground) : {p_sub.mean():.5f}")
print(f"  search time    : {infer_time:.0f}s / 600s budget")

np.savez_compressed("multistart_angles.npz", h=h_sub, gamma=g_sub, beta=b_sub, p_ground=p_sub)
print("saved multistart_angles.npz")

## 9. What this is for

Two things, and the second matters more.

**A score.** Compare `mean P(ground)` in §6 against notebook 02's direct-optimisation baseline.
The difference is exactly what buying breadth with the screen bought, since the Adam stage is the
same optimiser in both.

**Labels and starting points.** `multistart_angles.npz` holds the best angles this search could
find, which is the highest-quality supervision available for this task. Two ways to spend it:

- as a **candidate pool for the transformer's inference** — the elite-transfer idea of §5 applied
  to the model, replacing its uniform random rollout starts with points already known to be good;
- as **evaluation ground truth** — a per-instance ceiling to measure any model's gap against,
  instead of comparing means against a moving baseline.

**What it is not** is a solution to the stated task. The rules ask for a model that maps `h` to
angles, and score 20 points for the quality of that model; a search that reruns from scratch for
every instance earns none of them, and §7 shows full breadth does not fit the inference budget
anyway. Bank it as a safety submission, then beat it with something that has learned where to
look.